# Programming exercise 4: Coupled oscillators

Due on Monday, 11.05.2026, 20.00h

## The problem

Consider two coupled harmonic oscillators described by the Hamiltonian
$$
H=H_1 + H_2 + V =\frac{1}{2m}(p_1^2 + p_2^2) + \frac{1}{2}k(x_1^2 + x_2^2) + \frac{1}{2}\lambda(x_1 - x_2)^2
$$
We want to calculate the eigenvalues and eigenfunctions of this Hamiltonian and compare its dynamics to the dynamics of the corresponding classical problem. This problem is exactly solvable by transforming into center of mass and relative coordinates. However, as an exercise on how basis set expansion methods work, we want to treat the coupling term lambda as a perturbation and solve the problem by expanding into eigenfunctions of the uncoupled 2D harmonic oscillator and diagonalizing the resulting Hamiltonian.

In [ ]:
# load standard libraries

import numpy as np   # standard numerics library
import numpy.linalg as LA

import matplotlib.pyplot as plt   # for making plots

import time as time

import scipy.sparse.linalg as sLA
from scipy.sparse import coo_matrix

from ipywidgets import interactive, fixed

import Comp_Quant_Dynam as cqd # our numerics module

### Exercise 1

Transforming into center of mass and relative coordinates (normal modes)
$$
x_{CM}=(x_1+x_2)/2 \\
x_{rel}=(x_1-x_2)
$$ 
the Hamiltonian becomes
$$
H=\frac{p_{CM}^2}{2M}+\frac{p_{rel}^2}{2\mu} + \frac{1}{2}k_{CM}x_{CM}^2 + \frac{1}{2}k_{rel}x_{rel}^2 
$$
with $M=2m$, $\mu=m/2$, $k_{CM}=2k$, $k_{rel}=k/2+\lambda$. Thus the problem separates into two independent oscillators with frequencies 
$$
\omega_{CM} = \sqrt{\frac{k_{CM}}{M}} =\sqrt{\frac{k}{m}} =\omega \\
\omega_{rel} = \sqrt{\frac{k_{rel}}{\mu}} = \omega\sqrt{1+2\lambda/k}
$$
With this, one can write down the solution of the problem in the new coordinates as
$$
x_{CM}(t) = A_{CM} \cos(\omega t)+B_{CM} \sin(\omega t) \\
x_{rel}(t) = A_{rel} \cos(\omega_{rel} t)+B_{rel} \sin(\omega_{rel} t)
$$

Solve the classical problem for initial conditions $\{x_1(t=0)=x_{10}, x_2(t=0)=x_{20}, \dot{x}_1(t=0)=v_{10}, \dot{x}_2(t=0)=v_{20}\}$. For this, transform the initial conditions into CM and rel coordinates, determine the A,B coefficients and transform back to the original coordinates $x_1$, $x_2$. You don't have to document this calculation here but just plot the classical trajectories (i.e., plot the positions of the two particles as a function of time) for $m=k=1,\lambda=0.2$ and ${x_{10}=1,x_{20}=v_{10}=v_{20}=0}$ up to a time $t_f=50$. Observe the beating between center-of-mass and relative frequencies.

Optional: It is also instructive to make a density plot of the potential as a function of $x_1$ and $x_2$ and plot the classical trajectory on top of it.

Very instructive 2D plotting tips:
https://jakevdp.github.io/PythonDataScienceHandbook/04.04-density-and-contour-plots.html


In [ ]:
# plot the potential and the trajectories

L = 2
plotpoints = 100
lam = 0.2

# spatial grid
x, dx = cqd.utility.create_xvals(L, plotpoints)
y, dy = cqd.utility.create_xvals(L, plotpoints)
X, Y = np.meshgrid(x, y)

Z = cqd.hamiltonians.coupled_HO_potential(X, Y, lam)
tvec = cqd.utility.create_tvecs(200, 0.25)
# parameters for time evolution
x10, x20, v10, v20 = 1, 0, 0, 0
ini = [x10, x20, v10, v20] # initial conditions for the trajectory

x1, x2 = cqd.hamiltonians.class_traj(lam, ini, tvec)

# plotting 2D
fig = plt.figure()
p = plt.contourf(x, y, Z.transpose(), 20)
fig.colorbar(p)
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')

plt.plot(x1, x2, 'w')

# plot the trajectories in separate panels as a fuction of time
plt.figure()
plt.subplot(211)
plt.plot(tvec, x1)
#plt.xlabel('t')
plt.ylabel('$x_1$')
plt.subplot(212)
plt.plot(tvec, x2)
plt.xlabel('t')
plt.ylabel('$x_2$')

plt.show()


### Exercise 2

The exact eigenenergies can be read off from the Hamiltonian in CM and rel coordinates as the problem separates.
$$
E(n_{CM},n_{rel}) = (n_{CM}+1/2) \hbar \omega_{CM}+ (n_{rel}+1/2)\hbar \omega_{rel}
$$
The ground state energy is just $E_0 = E(0,0) = \frac{1}{2} \hbar (\omega_{CM}+\omega_{rel})=\frac{\hbar\omega}{2}(1+\sqrt{1+2\lambda/k})$.
We will use this analytical result to test the correctness and convergence of the basis set expansion approach we are going to employ now.

We want to expand the Hamiltonian in the eigenbasis of the 2D harmonic oscillator (use the same basis size $N_1=N_2=N_{max}$ for both oscillator modes 1 and 2. What is the resulting size of the basis set?). 
For this, one can expand the perturbation term $\frac{1}{2}\lambda(x_1 - x_2)^2$ into ladder operators and then apply it to the unperturbed basis states. Recall $x/x_{HO} = (a^\dagger + a)/\sqrt{2}$ and work in dimensionless units where $x_{HO}=1$.
How many off-diagonal elements will there be in each line? 

There are many ways of building the Hamiltonian matrix in the chosen basis. First, make sure you understand what the basis states are and how one would order them in a reasonable way.
A failsafe way to build up the Hamiltonian matrix is to iterate over all states of the chosen basis and determine the matrix elements it couples to. Another, arguably more elegant, way (which I recommend to use here) is to build the matrix representing the annihilation operator on the Hilbert space of one of the oscillators. The creation operator is then simply the transpose of that and the position operator can be built by adding the two. We have a 2D problem so we work in a Hilbert space that is a tensor product of the two Hilbert spaces of the oscillators. Thus, to represent the position operator $\hat{x}_1$ acting on oscillator 1 in the full Hilbert space  we have to use $\hat{x}_1 \otimes \mathbf{1}$, where $\mathbf{1}$ is the identity acting on the second oscillator. This can be achieved by using a Kronecker product. For multiplying or squaring operators you can use the numpy dot product function. Alternatively you can make use of the matrix multiplication operator @.
The second method has the advantage that it requires only to build one operator explicitly and the rest is matrix algebra (or two if you use different basis sizes for the two oscillators). Using sparse matrices this process can be very efficient.
Technical note of caution: The two described methods for building the Hamiltonian matrix might lead to slightly different results due to effects of the finite basis size. This will not affect the convergence of low-lying eigenstates.

Diagonalize the Hamiltonian and check your result by comparing the ground state energy to the exact result. How small can you make the basis such that the ground state energy is still converged? (You should find that $N_1=N_2=5$, i.e. 25 basis states, is sufficient to get the ground state with good precision up to $\lambda=2$.) Also look at some excited states, at least the lowest 6, make sure that your diagonalization results agree with the exact one, and understand the structure of the spectrum. Plot the eigenenergies for a grid of $\lambda$ values from 0 to 2.

Optional: As this problem can be solved analytically, we are complicating it by doing the basis set expansion. However, one could now use this method to apply it to non-integrable cases like a quartic (non-linear) coupling or even the quantum version of the Henon-Heiles problem, which does not separate and is classically chaotic!
If you want to go beyond, try other potentials!


In [ ]:
# build the Hamiltonian (the manual way):
N1 = N2 = 4
lam = 0.2

H_mat_man = cqd.hamiltonians.build_H_coupled_HO_man(N1, N2, lam)
evals_man, evecs_man = sLA.eigsh(H_mat_man, k=10, which='SA')

print("Eigenvalues manual method:\n", evals_man)

In [ ]:
#build the Hamiltonian (the smarter way):

N1 = N2 = 4
lam = 0.2

H_mat_impr = cqd.hamiltonians.build_H_coupled_HO_improved(N1, N2, lam)
evals_impr, evecs_impr = sLA.eigsh(H_mat_impr, k=10, which='SA')

print("Eigenvalues improved method:\n", evals_impr)

In [ ]:
print("Difference in eigenvalues:\n", evals_man - evals_impr)

In [ ]:
# Now loop over lam (use the improved implementation here)
N1 = N2 = 8
kmax = 10
lamvec = np.linspace(0,1.9,100)

EvalsAll = np.zeros((len(lamvec),kmax))

for i in range(len(lamvec)):
    H_mat = cqd.hamiltonians.build_H_coupled_HO_improved(N1, N2, lamvec[i])
    evals, evecs = sLA.eigsh(H_mat, k=kmax, which='SA')
    EvalsAll[i] = evals
    
#print(EvalsAll[:,0])

In [ ]:
plt.plot(lamvec, cqd.hamiltonians.coupled_HO_E0_exact(lamvec), label ='exact')
plt.plot(lamvec, EvalsAll[:, 0], 'k--', label = 'numerical basis expansion')
plt.xlabel('$\\lambda$')
plt.ylabel('$E_0$')
plt.legend()
plt.show()

In [ ]:
# more states of the spectrum

n_cm_rel_vec = np.array([[0,0],[0,1],[1,0],[2,0]])
En_exact_all = np.zeros((len(lamvec),4))
for j in range(4):
    n_cm_j = n_cm_rel_vec[j,0]
    n_rel_j = n_cm_rel_vec[j,1]
    En_exact_all[:,j] = cqd.hamiltonians.coupled_HO_eigenenergies_exact(n_cm_j, n_rel_j, lamvec)
plt.plot(lamvec,En_exact_all,'k--')
plt.plot(lamvec, EvalsAll[:, 0:4])
plt.xlabel('$\\lambda$')
plt.ylabel('$E_0$')
plt.title('exact (dashed) and ED result (solid) ')
plt.show()

In [ ]:
H_mat_2 = cqd.hamiltonians.build_H_coupled_HO_improved(N1, N2, lam)
evals2, evecs2 = sLA.eigsh(H_mat_2, k=50, which='SA')

### Exercise 3 (optional)

Calculate the time evolution for different initial conditions (as on exercise sheet 2) using the eigenstates obtained in exercise 2. Try the following initial states: 1) oscillator 1 in the first excited state, oscillator two in its ground state $|0\rangle$; 2) oscillator 1 in $|\psi_0\rangle = (|0\rangle+|1\rangle)/\sqrt(2)$, oscillator two in state $|0\rangle$;  3) oscillator 1 in a coherent state (e.g. with $\langle n \rangle = |\alpha|^2 = 2$) and oscillator 2 in $|0\rangle$. Example parameters: $\lambda=0.2$, $N_1=N_2=10$, $t_f = 40$.

Monitor the dynamics by calculating $\langle x_1 \rangle$ and $\langle x_2 \rangle$. Compare your observations to the classical expectation: propagate a classical particle with $x_{i0}=\langle x_i(t=0) \rangle$ and $p_{i0}=\langle p_i(t=0) \rangle$. Describe and interpret your observations.

Calculate also the energy expectation value of each oscillator, as well as the probabilities for each oscillator to be in state $|n\rangle$. You should observe that the energy fluctuates between the oscillators.

Represent the wave function on a 2D spatial grid using the exact harmonic oscillator eigenfunctions (Hermite Polynomials...). Animate the time-dependence of the wave packet in a 2D density plot.

In [ ]:
N1 = N2 = 10
lam = 0.2

dim = N1 * N2
N_vec = [N1, N2]

evals_H1 = np.arange(N1) + 0.5
evals_H2 = np.arange(N2) + 0.5
H1 = cqd.operators.diagonal_op_sparse(evals_H1)
H2 = cqd.operators.diagonal_op_sparse(evals_H2)

H1_full = cqd.operators.n_party_op_sparse(N_vec, 0, H1)
H2_full = cqd.operators.n_party_op_sparse(N_vec, 1, H2)

x1_local = cqd.operators.x_operator_sparse(N1)
x2_local = cqd.operators.x_operator_sparse(N2)

p1_local = cqd.operators.p_operator_sparse(N1)
p2_local = cqd.operators.p_operator_sparse(N2)

x1_full = cqd.operators.n_party_op_sparse(N_vec, 0, x1_local)
x2_full = cqd.operators.n_party_op_sparse(N_vec, 1, x2_local)

p1_full = cqd.operators.n_party_op_sparse(N_vec, 0, p1_local)
p2_full = cqd.operators.n_party_op_sparse(N_vec, 1, p2_local)
    
# build H and diagonalize
H_mat = cqd.hamiltonians.build_H_coupled_HO_improved(N1, N2, lam)
evals, evecs = sLA.eigsh(H_mat, k=50, which='SA')

### various different initial conditions:

## initial state in bare basis: left in 1st exc state, second in gs
# index = cqd.utility.state2idx(N1, N2, [1, 0])
# ini = np.eye(1,dim,index)[0]

## initially oscillator 1 is in a superposition
index1 = cqd.utility.state2idx(N1, N2, [0, 0]);
index2 = cqd.utility.state2idx(N1, N2, [1, 0])
ini = (np.eye(1, dim, index1)[0] + np.eye(1, dim, index2)[0])/np.sqrt(2) 

## oscillator 1 initially in coherent state
#alpha = np.sqrt(2);
#ini1 = cqd.utility.create_coherent_state(N1, alpha)
#ini2 = np.eye(1, N2, 0)
#ini = np.kron(ini1, ini2)[0] # initial state is the product of the coherent state and the n = 4 state.

# calculate projections on eigenstates
ini_coeffs_eigenbasis = cqd.unitaries.init_coeffs_eigenbasis(ini, evecs)

print('Initial state properties:')
print("Norm: ", LA.norm(ini_coeffs_eigenbasis))
print("⟨x₁⟩: ", cqd.utility.expectation_value(ini, x1_full))
print("⟨p₁⟩: ", cqd.utility.expectation_value(ini, p1_full))

# some testing: -> There is a subtle difference to the manual method: for the diagonal elments with i1=N1 or i2=N2, 
# the diagonal is not n+1/2 because the in (a^dag a + a a^dag) the second term is zero (in the brute force method) 
# because it would lead outside of the Hilbert space. In principle this would also be the case for the unperturbed part.
# This is an issue of the truncation and both approaches are viable.

#print(H_mat_man - H_mat_impr)
#plt.imshow((H_mat_man-H_mat_impr).toarray())
#print(H_mat_impr.diagonal())
#print(H_mat_man.diagonal())
#print(x1_full)
#plt.show()

# calculate observables
t_vec = cqd.utility.create_tvecs(200, 0.25)
observables = np.zeros((5,len(tvec)), dtype=complex)
for i in range(len(tvec)):
    psi_t = cqd.unitaries.t_evol_eigenbasis(ini_coeffs_eigenbasis, tvec[i], evals, evecs)

    observables[0,i] = cqd.utility.expectation_value(psi_t, x1_full)
    observables[1,i] = cqd.utility.expectation_value(psi_t, x2_full)
    observables[2,i] = cqd.utility.expectation_value(psi_t, H1_full)
    observables[3,i] = cqd.utility.expectation_value(psi_t, H2_full)
    observables[4,i] = cqd.utility.expectation_value(psi_t, H_mat)

# calculate probabilities in single osc. states
Pn1 = 1j*np.zeros((N1,len(tvec)))
Pn2 = 1j*np.zeros((N2,len(tvec)))
for i in range(len(tvec)):
    psi_t = cqd.unitaries.t_evol_eigenbasis(ini_coeffs_eigenbasis, tvec[i], evals, evecs)
    for n in range(N1):
        pn1 = cqd.operators.n_proj_operator_sparse(N_vec, 0, n)
        pn2 = cqd.operators.n_proj_operator_sparse(N_vec, 1, n)
        Pn1[n,i] = cqd.utility.expectation_value(psi_t, pn1)
        Pn2[n,i] = cqd.utility.expectation_value(psi_t, pn2)


assert np.all(np.abs(observables.imag) < 1e-10) # To make sure all imaginary parts are indeed zero.
observables = observables.real # to get rid of warnings from the plot command. 
Pn1 = Pn1.real
Pn2 = Pn2.real

plt.plot(tvec,observables[0])    
plt.plot(tvec,observables[1])
plt.plot(tvec,observables[2])
plt.plot(tvec,observables[3])
plt.plot(tvec,observables[4])
plt.xlabel('t')
plt.ylabel('observables')
plt.legend(('$\\langle x_1\\rangle $','$\\langle x_2\\rangle $','$\\langle H_1\\rangle $','$\\langle H_2\\rangle $','$\\langle H_{tot}\\rangle $'))
plt.legend(('$x_1$','$x_2$','$H_1$','$H_2$','$H_{tot}$'))

plt.show()

plt.figure(figsize=(10,4))
plt.subplot(121)
plt.imshow(Pn1, extent=[0, tvec[-1], -.5, N1 - 0.5], origin='lower',
           aspect='auto',
           interpolation='none')
plt.xlabel('t')
plt.ylabel('$n_1$')
plt.colorbar()

plt.subplot(122)
plt.imshow(Pn2, extent=[0, tvec[-1], -.5, N2 - 0.5], origin='lower',
           #cmap='RdGy',
           aspect='auto',
           interpolation='none')
plt.xlabel('t')
plt.ylabel('$n_2$')
plt.colorbar()
plt.show()


Observe that the energy always oscillates between the two oscillators. In the case of each oscillator starting in an eigenstate, the spatial mean stays zero all the time. Otherwise the mean oscillates similar to the classical case.

In [ ]:
# compare to the classical trajectories (coherent state: initially at rest at <x>)
x10, x20, v10, v20 = observables[0,0], observables[1,0], 0, 0 # p1 and p2 are zero for all three states we considered

ini = [x10, x20, v10, v20]
x1, x2 = cqd.hamiltonians.class_traj(lam, ini, tvec)

plt.figure(figsize=(12,5))
plt.subplot(211)
plt.plot(tvec,observables[0])    
plt.plot(tvec,x1,'k--')
plt.xlabel('t')
plt.ylabel('$x_1$')
plt.legend(('quantum','classical'))
plt.subplot(212)
plt.plot(tvec,observables[1])    
plt.plot(tvec,x2,'k--')
plt.xlabel('t')
plt.ylabel('$x_2$')
plt.legend(('quantum','classical'))
plt.show()


The spatial mean in fact always exactly matches the classical case if we use the $x_i$ and $p_i$ of the initial state as initial conditions for the classical evolution! This is in general true for bosonic modes with linear coupling (Hamiltonian is quadratic in the fields).

In [ ]:
# calculate position wave functions

# make a spatial grid
L=6
plotpoints = 101
xgrid, dx = cqd.utility.create_xvals(L, plotpoints)

# calculate the HO eigenfunctions on this grid
basis_state_pos = cqd.hamiltonians.HO_product_eigenstates(N1, N2, xgrid)

# calculate the 2D evolution of the probability on the spatial grid
wfcts = np.zeros((len(tvec), len(xgrid), len(xgrid)), dtype=complex)
for i in range(len(tvec)):
    psi_t = cqd.unitaries.t_evol_eigenbasis(ini_coeffs_eigenbasis, tvec[i], evals, evecs)
    for k in range(dim):
        wfcts[i] += psi_t[k] * basis_state_pos[k]



In [ ]:
# make an interactive density plot    
interactive_plot = interactive(cqd.plotting.plot_prob_amplitude_2D, t=(0, len(tvec)-1), wfcts=fixed(wfcts), tvec=fixed(tvec), L=fixed(L))
interactive_plot


In [ ]:
N1 = N2 = 15
lam = 5.

#Hmat2 = buildH2(N1,N2,lam)
H_mat_man_plt = cqd.hamiltonians.build_H_coupled_HO_improved(N1, N2, lam).toarray()
evals_plot, evecs_plot = LA.eigh(H_mat_man_plt)

# calculate position wave functions

# make a spatial grid
L=6
plotpoints = 101
xgrid, dx = cqd.utility.create_xvals(L, plotpoints)
dim = N1*N2

# calculate the product HO eigenfunctions on this grid
basis_state_pos = cqd.hamiltonians.HO_product_eigenstates(N1, N2, xgrid)

In [ ]:
# Plotting the eigenstates of the coupled HO 

eigenstate_idx = 44

psi = evecs_plot[:,eigenstate_idx]

gs_wfct = np.zeros((len(xgrid), len(xgrid)), dtype=complex)

for k in range(dim):
    gs_wfct += psi[k] * basis_state_pos[k]

fig = plt.figure()
ax = fig.add_subplot()
ax.imshow(np.abs(gs_wfct) ** 2, extent = (-L/2, L/2, -L/2, L/2), interpolation='none', origin='lower')
# add labels and legends
ax.set_xlabel("$x_2$")
ax.set_ylabel("$x_1$")
plt.show()